In [1]:
import cProfile
import json
import pstats
import sys
from dataclasses import asdict
from datetime import datetime
from pathlib import Path
from pickle import load
from time import perf_counter

import pandas as pd
from lightgbm import LGBMClassifier
from onnxmltools.convert.lightgbm.operator_converters.LightGbm import convert_lightgbm
from skl2onnx import convert_sklearn, update_registered_converter
from skl2onnx.common.data_types import FloatTensorType
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from sklearn.pipeline import Pipeline

# Racine du projet : OC-Projet-8/ (2 niveaux au-dessus de src/notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks" and ROOT.parent.name == "src":
    ROOT = ROOT.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

#BASE_DIR = Path(__file__).resolve().parents[2]
DATA_PATH = ROOT / "data" / "original" / "demonstration_data.csv"
SCHEMA_PATH = ROOT / "data" / "schema" / "typeAdapters.json"
MODEL_PATH = ROOT / "src" / "model" / "lgb_model.pkl"
PARAMS_PATH = ROOT / "data" / "profils.json"

In [3]:
from src.utils.utils import business_cost, custom_sampler_ratio  # noqa: F401
from src.api import scoring_api
from src.database.simulate_client import random_datetime_between, simulate_profils
from src.monitoring.data import (
    compute_kpis,
    default_date_range,
    load_logs,
)

In [4]:
start, end = default_date_range()

## Latences enregistrées en base

In [5]:
full_data = load_logs(start, end)

### Volume et latences globales

In [6]:
kpis = compute_kpis(full_data)
pd.Series(kpis)

total         5005.000000
success       3995.000000
errors        1010.000000
error_rate       0.201798
p50_ms          13.720000
p95_ms          37.033600
dtype: float64

- total: nombre total de requêtes
- success: nombre de requêtes correctes ayant abouties à une prédiction
- errors: nombre de requêtes erronées ayant été rejetées par l'API (inputs incorrects)
- p50_ms: latence globale médiane
- p95_ms: latence globale des 5% de requêtes les plus longues

In [7]:
success = full_data.loc[~full_data["error"]]
errors = full_data.loc[full_data["error"]]

### Latence — requêtes réussies

`execution_time_ms` mesure le **temps total** côté API : validation + inférence + écriture PostgreSQL.

In [8]:
success.describe().execution_time_ms

count    3995.000000
mean       19.940071
std        62.643793
min         8.754000
25%        12.710500
50%        14.770000
75%        20.731000
max      2931.767000
Name: execution_time_ms, dtype: float64

In [9]:
outliers = (len(success.loc[success["execution_time_ms"] > 100])/3901)*100
print(f"Pourcentage de requêtes longues (>100ms) : {outliers:.2f}%")

Pourcentage de requêtes longues (>100ms) : 0.13%


Médiane à 13ms, moyenne à 21ms. La moyenne semble tirée vers le haut par un petit nombre de valeurs extrêmes.

### Latence — requêtes échouées

In [10]:
errors.describe().execution_time_ms

count    1010.000000
mean        1.586962
std         4.516681
min         0.000000
25%         0.000000
50%         0.000000
75%         2.160500
max       130.715000
Name: execution_time_ms, dtype: float64

La latence des erreurs dépend de l'étape du pipeline ayant rejeté l'input et donc du type d'erreur.

In [11]:
errors.error_message.unique()

array(['No features value was recieved from the user.',
       'Unknown feature, not suported by pydantic validation : invalid_feature'],
      dtype=object)

In [12]:
for error in errors.error_message.unique():
    print(error)
    print(errors.loc[errors["error_message"] == error].describe().execution_time_ms)
    print("\n")


No features value was recieved from the user.
count    509.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: execution_time_ms, dtype: float64


Unknown feature, not suported by pydantic validation : invalid_feature
count    501.000000
mean       3.199265
std        5.999960
min        1.167000
25%        1.826000
50%        2.167000
75%        3.214000
max      130.715000
Name: execution_time_ms, dtype: float64




On observe que deux messages d'erreurs sont majoritairement représentés dans la base: aucune feature reçue et feature inconnue. La première entraine un rejet de l'input avant tout traitement et renvoie donc une latence égale à 0ms. La seconde montre une moyenne et une médiane à 2ms. Quelques latence plus importantes existent toutefois, avec un maximum à 11ms.

**Interprétation :** Les succés (~13 ms P50) sont beaucoup plus coûteux (**~10× plus**) que le chemin d'erreur (~1,5 ms P50). L'inférence via la fonction infer_from_new_vector explique donc probablement le temps de réponse plus important dans le cas d'une requête valide.

## Profiling cProfile

In [29]:
N_FEATURES = 262 #Nb de features attendues par le model
N_PROFILE_CALLS = 500
EVENT_TIMES = [random_datetime_between(start, end).isoformat() for ndate in range(N_PROFILE_CALLS)]
SIMULATE_PROFILS = True

In [30]:
#Tableau de données de démonstration
data = pd.read_csv(DATA_PATH, sep = ";")

#Modèle de scoring
with open(MODEL_PATH, "rb") as f:
    scoring_model = load(f)

#Dictionnaire de nouveaux paramètres

if SIMULATE_PROFILS:
    simulate_profils(ROOT, N_PROFILE_CALLS, 0) #On simule 500 nouveaux clients corrects 

with open(PARAMS_PATH) as f:
    params_list = json.load(f)


Simulating profils...
New profils registered. 



In [31]:
update_registered_converter(
    LGBMClassifier, 
    "LightGbmLGBMClassifier",
    calculate_linear_classifier_output_shapes, 
    convert_lightgbm,
    options={"nocl": [True, False], "zipmap": [True, False, "columns"]}
)

inference_steps = [
    ('imputer', scoring_model.estimator.named_steps['simpleimputer']), 
    ('transformer', scoring_model.estimator.named_steps['powertransformer']),
    ('scaler', scoring_model.estimator.named_steps['minmaxscaler']), 
    ('classifier', scoring_model.estimator.named_steps['lgbmclassifier']),
]

sklearn_pipe = Pipeline(inference_steps)

# 3. Définir le type d'entrée et convertir
initial_type = [('float_input', FloatTensorType([None, N_FEATURES]))]

target_opsets = {
    '': 15,          # Domaine par défaut (ai.onnx)
    'ai.onnx.ml': 3  # Domaine Machine Learning (LightGBM, Arbres, etc.)
}

onnx_model = convert_sklearn(sklearn_pipe, initial_types=initial_type, target_opset=target_opsets)

In [32]:
SAVING_DIR = ROOT / "src" / "model"
with open(SAVING_DIR / "onnx_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [33]:
#MODEL = scoring_model
MODEL = SAVING_DIR / "onnx_model.onnx"

In [34]:
def run_validate_params(params: dict):
    params_df = pd.DataFrame(
        {
            "feature": list(params.keys()),
            "value": list(params.values()),
        }
    )
    
    scoring_api.validate_params(params_df, scoring_api.explicative_features)

def run_inference(params: dict, event_time: datetime):
    scoring_api.infer_from_new_vector(
        params = params,
        model = MODEL,
        start_time=perf_counter(),
        event_time = event_time,
        persist = False,
        onnx = True
        )

def run_pipeline(params: dict, event_time: str):
    scoring_api.process_scoring_request(
        user_values = params,
        model = MODEL,
        simulated_event_time = event_time,
        persist = True,
        onnx = True
        )

In [35]:
scenarios = [
    {
        "key":"A",
        "title":"Validation Pydantic",
        "description":"`validate_params` seul (sans inférence ni DB).",
        "function": run_validate_params,
    },
    {
        "key":"B",
        "title":"Inférence sans DB",
        "description":"`infer_from_new_vector` sans écriture PostgreSQL.",
        "function":run_inference,
    },
    {
        "key":"C",
        "title":"Chemin API complet",
        "description":"`process_scoring_request` (validation + inférence + DB).",
        "function":run_pipeline,
    },
]

### Profilage modèle sklearn

In [36]:
profilers = {}
for s in scenarios:
    profiler = cProfile.Profile()
    profiler.enable()
    for index in range(N_PROFILE_CALLS):
        if s['key'] == 'A':
            s['function'](params_list[index])
        elif s['key'] == 'B':
            s['function'](params_list[index], datetime.fromisoformat(EVENT_TIMES[index]))
        else:
            s['function'](params_list[index], EVENT_TIMES[index])
    profiler.disable()
    profilers[s['key']] = profiler

Exception ignored When destroying _lsprof profiler:
Traceback (most recent call last):
  File "C:\Users\CS\AppData\Local\Temp\ipykernel_16500\2408743531.py", line 4, in <module>
RuntimeError: Cannot install a profile function while another profile function is being installed


In [37]:
RESULT_COLUMNS = [
    "scenario",
    "filename",
    "func",
    "ncalls",
    "tottime",
    "percall_tottime",
    "cumtime",
]

scenario_titles = {s["key"]: s["title"] for s in scenarios}

frames = []
avg_ms = []
for index, key in enumerate(profilers):
    p = pstats.Stats(profilers[key]).sort_stats('cumulative')
    p.dump_stats(ROOT / "data" / f'profile_{key}.pstats')
    avg_ms.append(1000*(p.total_tt/N_PROFILE_CALLS))
    print(f"Durée moyenne (ms) - {scenario_titles[key]} : {avg_ms[index]}")

    df = pd.DataFrame(
        [
            {"func": func, **asdict(stats)}
            for func, stats in p.get_stats_profile().func_profiles.items()
        ]
    )
    df = df[~df["func"].str.contains("run", na=False)] #exclue les wrappers du notebook
    #top = df.sort_values("cumtime", ascending=False).head(15).copy()
    top = df.head(15).copy()
    top["scenario"] = scenario_titles[key]
    top["filename"] = top["file_name"].apply(lambda path: Path(path).name)
    frames.append(top[RESULT_COLUMNS])

profile_results = pd.concat(frames, ignore_index=True)

Durée moyenne (ms) - Validation Pydantic : 3.2924187999999974
Durée moyenne (ms) - Inférence sans DB : 12.746236600000001
Durée moyenne (ms) - Chemin API complet : 18.640677800000013


In [38]:
print(f"La validation représente {round((avg_ms[0]/avg_ms[2])*100)}% du temps complet de traitement")
print(f"L'inférence représente {round(((avg_ms[1]-avg_ms[0])/avg_ms[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms[1] - avg_ms[0])}ms")
print(f"L'enregistrement en base de données représente {round(((avg_ms[2]-avg_ms[1])/avg_ms[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms[2]- avg_ms[1])}ms")

La validation représente 18% du temps complet de traitement
L'inférence représente 51% du temps complet de traitement avec une durée moyenne de 9ms
L'enregistrement en base de données représente 32% du temps complet de traitement avec une durée moyenne de 6ms


In [39]:
profile_results.loc[profile_results.scenario == "Validation Pydantic"].iloc[0:4]

,scenario,filename,func,ncalls,tottime,percall_tottime,cumtime
0,Validation Pydantic,scoring_api.py,validate_params,500,0.028,0.0,1.465
1,Validation Pydantic,managers.py,__init__,500,0.000,0.0,0.000
2,Validation Pydantic,frame.py,iterrows,5099,0.021,0.0,0.572
3,Validation Pydantic,type_adapter.py,_init_core_attrs,4599,0.052,0.0,0.494


Le coût en temps de traitement lors de la validation des paramètres est dominé par la création répétée de `TypeAdapter` et `DataFrame.iterrows` dans `validate_params` (boucle for par ligne du dictionnaire params).

In [40]:
profile_results.loc[profile_results.scenario == 'Inférence sans DB']

,scenario,filename,func,ncalls,tottime,percall_tottime,cumtime
15,Inférence sans DB,scoring_api.py,infer_from_new_vector,500,0.021,0.000,5.827
16,Inférence sans DB,managers.py,__init__,1000,0.000,0.000,0.000
17,Inférence sans DB,onnxruntime_inference_collection.py,_create_inference_session,500,1.400,0.003,4.900
18,Inférence sans DB,construction.py,arrays_to_mgr,500,0.003,0.000,0.278
19,Inférence sans DB,base.py,reindex,500,0.003,0.000,0.083
20,Inférence sans DB,generic.py,_reindex_axes,500,0.003,0.000,0.212
21,Inférence sans DB,construction.py,nested_data_to_arrays,500,0.002,0.000,0.147
22,Inférence sans DB,construction.py,to_arrays,500,0.003,0.000,0.143
23,Inférence sans DB,managers.py,create_block_manager_from_column_arrays,500,0.003,0.000,0.133
24,Inférence sans DB,generic.py,_reindex_with_indexers,500,0.005,0.000,0.125


En ajoutant l'inférence, ce sont les fonction sklearn/imbalance-learn qui apparaissent les plus couteuses.

In [41]:
profile_results.loc[profile_results.scenario == 'Chemin API complet'].iloc[0:10]

,scenario,filename,func,ncalls,tottime,percall_tottime,cumtime
30,Chemin API complet,scoring_api.py,process_scoring_request,500,0.558,0.001,9.318
31,Chemin API complet,scoring_api.py,infer_from_new_vector,500,0.031,0.000,6.966
32,Chemin API complet,onnxruntime_inference_collection.py,__init__,500,0.000,0.000,0.000
33,Chemin API complet,onnxruntime_inference_collection.py,_create_inference_session,500,1.456,0.003,5.112
34,Chemin API complet,scoring_api.py,validate_params,500,0.030,0.000,1.558
35,Chemin API complet,database.py,save_prediction_log,500,0.010,0.000,0.967
36,Chemin API complet,frame.py,iterrows,5099,0.023,0.000,0.599
37,Chemin API complet,type_adapter.py,_init_core_attrs,4599,0.057,0.000,0.526
38,Chemin API complet,base.py,execute,500,0.002,0.000,0.516
39,Chemin API complet,elements.py,_execute_on_connection,500,0.001,0.000,0.514


La même conclusion qu'en scénario B apparait sur le pipeline complet.

### Profilage model onnx

In [42]:
profilers_onnx = {}
for s in scenarios:
    profiler = cProfile.Profile()
    profiler.enable()
    for index in range(N_PROFILE_CALLS):
        if s['key'] == 'A':
            s['function'](params_list[index])
        elif s['key'] == 'B':
            s['function'](params_list[index], datetime.fromisoformat(EVENT_TIMES[index]))
        else:
            s['function'](params_list[index], EVENT_TIMES[index])
    profiler.disable()
    profilers_onnx[s['key']] = profiler

In [43]:
RESULT_COLUMNS = [
    "scenario",
    "filename",
    "func",
    "ncalls",
    "tottime",
    "percall_tottime",
    "cumtime",
]

scenario_titles = {s["key"]: s["title"] for s in scenarios}

frames_onnx = []
avg_ms_onnx = []
for index, key in enumerate(profilers_onnx):
    p = pstats.Stats(profilers_onnx[key]).sort_stats('cumulative')
    p.dump_stats(ROOT / "data" / f'profile_{key}_onnx.pstats')
    avg_ms_onnx.append(1000*(p.total_tt/N_PROFILE_CALLS))
    print(f"Durée moyenne (ms) - {scenario_titles[key]} : {avg_ms_onnx[index]}")

    df = pd.DataFrame(
        [
            {"func": func, **asdict(stats)}
            for func, stats in p.get_stats_profile().func_profiles.items()
        ]
    )
    df = df[~df["func"].str.contains("run", na=False)] #exclue les wrappers du notebook
    #top = df.sort_values("cumtime", ascending=False).head(15).copy()
    top = df.head(15).copy()
    top["scenario"] = scenario_titles[key]
    top["filename"] = top["file_name"].apply(lambda path: Path(path).name)
    frames_onnx.append(top[RESULT_COLUMNS])

profile_results_onnx = pd.concat(frames_onnx, ignore_index=True)

Durée moyenne (ms) - Validation Pydantic : 3.5044794000000015
Durée moyenne (ms) - Inférence sans DB : 12.48221300000001
Durée moyenne (ms) - Chemin API complet : 19.468231399999993


In [25]:
print(f"La validation représente {round((avg_ms_onnx[0]/avg_ms_onnx[2])*100)}% du temps complet de traitement")
print(f"L'inférence représente {round(((avg_ms_onnx[1]-avg_ms_onnx[0])/avg_ms_onnx[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms_onnx[1] - avg_ms_onnx[0])}ms")
print(f"L'enregistrement en base de données représente {round(((avg_ms_onnx[2]-avg_ms_onnx[1])/avg_ms_onnx[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms_onnx[2]- avg_ms_onnx[1])}ms")

La validation représente 17% du temps complet de traitement
L'inférence représente 66% du temps complet de traitement avec une durée moyenne de 12ms
L'enregistrement en base de données représente 17% du temps complet de traitement avec une durée moyenne de 3ms
